# PAI Lab 11 — Generative AI Concepts
**Task 1: Explain the difference between key AI concepts**

---

This notebook goes through 8 concepts that keep coming up in modern AI: LangChain, LLMs, RAG, FAISS, Vectors, VectorDB, GenAI, and GANs. I'll explain each one simply, with a code example where it makes sense.

## 1. LangChain

LangChain is a Python framework for building apps on top of language models. On its own, an LLM just takes a prompt and returns text. LangChain lets you connect it to other things — databases, APIs, your own files, memory of past messages — without writing all the plumbing yourself.

Think of it like this: the LLM is the brain, LangChain is the body that lets it actually do things.

The main building blocks:
- **Chains** — run steps in order (prompt → LLM → clean the output)
- **Agents** — let the LLM decide which tool to call next
- **Memory** — remember what the user said earlier
- **Retrievers** — pull relevant text from a database before answering

Most RAG-based chatbots are built with LangChain because it saves a lot of time.

In [ ]:
# A simple LangChain-style chain — no API key needed, just showing the idea

def build_prompt(context, question):
    return f"""Use the context below to answer the question.

Context: {context}
Question: {question}
Answer:"""

def fake_llm(prompt):
    # In real code, this calls GPT or LLaMA
    return "[LLM would answer here based on the context]"

def chain(context, question):
    prompt = build_prompt(context, question)
    reply  = fake_llm(prompt)
    return reply

ctx = "Dubai Boys Hostel is in Lahore. Monthly rent is Rs. 12,000."
q   = "Where is the hostel and how much is the rent?"

print(chain(ctx, q))

## 2. LLMs — Large Language Models

An LLM is a model trained on a huge amount of text — books, websites, code, basically the internet. It learns patterns well enough that it can write sentences, answer questions, translate, summarize, and generate code.

The basic idea: given some text, predict what word comes next. Do that billions of times across billions of examples, and you get a model that seems to "understand" language.

Some well-known LLMs:
- **GPT-4** (OpenAI) — the one behind ChatGPT
- **LLaMA** (Meta) — open source, can run on your own machine
- **Claude** (Anthropic) — good at long documents
- **Gemini** (Google) — works with text and images together

One thing to keep in mind: LLMs only know what was in their training data. They can't look anything up on their own. That's the problem RAG was built to fix.

In [ ]:
# GPT-2 is a small LLM that runs locally — good for demos
# pip install transformers torch

from transformers import pipeline

gen  = pipeline("text-generation", model="gpt2", max_new_tokens=50)
seed = "Artificial intelligence is changing how we"
out  = gen(seed)[0]["generated_text"]

print(out)
# Note: GPT-2 is from 2019 and pretty small.
# Modern models like GPT-4 produce much better output.

## 3. RAG — Retrieval-Augmented Generation

RAG solves the main weakness of LLMs: they don't know your private data, and their knowledge has a cutoff date.

The fix is simple: before the LLM answers, search your documents for relevant content, then pass that content to the LLM along with the question. The LLM answers based on what it actually retrieved, not what it memorized.

**RAG = LLM + a search step**

How it works, step by step:
1. Your documents get split into chunks and turned into vectors
2. Those vectors get stored in a VectorDB
3. User asks a question → question gets turned into a vector
4. VectorDB finds the most similar chunks
5. Those chunks + the question go to the LLM
6. LLM answers using the retrieved content

This is how most production chatbots work today. Without RAG, the LLM is just guessing.

In [ ]:
# A minimal RAG pipeline using simple keyword matching as the retriever

docs = [
    "Dubai Boys Hostel is at Raiwind Rd, Lahore, Pakistan.",
    "Standard room rent is Rs. 12,000 per month.",
    "Check-in is at 12 PM. One month advance is required.",
    "Free Wi-Fi is available throughout the hostel.",
    "Canteen serves breakfast for Rs. 150 and dinner for Rs. 200.",
]

def retrieve(query, top=2):
    words  = set(query.lower().split())
    scored = [(sum(w in d.lower() for w in words), d) for d in docs]
    scored.sort(reverse=True)
    return [d for _, d in scored[:top]]

def rag(query):
    hits    = retrieve(query)
    context = "\n".join(hits)
    prompt  = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    print("Retrieved docs:")
    for h in hits:
        print(" -", h)
    print("\nPrompt sent to LLM:")
    print(prompt)

rag("What is the rent and check-in time?")

## 4. FAISS

FAISS (Facebook AI Similarity Search) is a library that finds similar vectors fast. It was built by Meta for cases where you have millions of vectors and need to search them in milliseconds.

Without something like FAISS, you'd have to compare your query vector to every single stored vector one by one. That's fine for 100 documents. It falls apart at 1 million.

FAISS builds an **index** ahead of time — a smart data structure that lets it skip most comparisons and still return accurate results. It's the search engine sitting inside your RAG pipeline.

The most common index for small projects is `IndexFlatL2` — it does exact search using Euclidean distance. For larger datasets, `IndexIVFFlat` approximates the search to go faster.

In [ ]:
# pip install faiss-cpu numpy
import numpy as np
import faiss

dim  = 8   # real embeddings are 384-1536 dims, keeping it small here

np.random.seed(42)
doc_vecs = np.random.rand(5, dim).astype("float32")

labels = [
    "Location info",
    "Room prices",
    "Check-in policy",
    "Wi-Fi info",
    "Canteen prices",
]

# Build index and add all vectors
idx = faiss.IndexFlatL2(dim)
idx.add(doc_vecs)
print(f"Vectors stored: {idx.ntotal}")

# Search with a random query vector
query      = np.random.rand(1, dim).astype("float32")
dists, ids = idx.search(query, k=2)

print("\nTop 2 closest documents:")
for i, d in zip(ids[0], dists[0]):
    print(f"  {labels[i]}  (distance: {d:.4f})")

## 5. Vector (Embedding)

A vector is just a list of numbers. In AI, we use vectors to represent words, sentences, or images in a way that captures meaning.

The useful thing is that similar meanings end up as similar numbers. "King" and "queen" will have vectors that are close together. "King" and "pizza" will be far apart.

```
"king"   →  [0.21, 0.85, -0.34, 0.76, ...]
"queen"  →  [0.20, 0.83, -0.30, 0.74, ...]  ← very close
"pizza"  →  [-0.50, 0.10, 0.88, -0.21, ...]  ← far away
```

This is how search works in RAG — the query gets turned into a vector, and we find the stored vectors that are closest to it.

You measure closeness between vectors using **cosine similarity** (1.0 = same direction, 0.0 = unrelated).

In [ ]:
import numpy as np

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Toy vectors — real ones from BERT have 768 dimensions
king  = np.array([0.9,  0.8,  0.1,  0.2])
queen = np.array([0.85, 0.78, 0.15, 0.25])
pizza = np.array([0.1,  0.05, 0.9,  0.85])

print(f"king  vs queen : {cosine_sim(king, queen):.4f}  (high — similar meaning)")
print(f"king  vs pizza : {cosine_sim(king, pizza):.4f}  (low — unrelated)")

# The classic word vector trick
man   = np.array([0.88, 0.75, 0.12, 0.18])
woman = np.array([0.83, 0.70, 0.20, 0.28])
result = king - man + woman

print(f"\nking - man + woman = {result.round(2)}")
print(f"Similarity to queen: {cosine_sim(result, queen):.4f}")

## 6. VectorDB

A VectorDB is a database built specifically for storing and searching vectors. A regular database lets you search by exact values (`WHERE name = 'Ali'`). A VectorDB searches by similarity — give it a query vector, it returns the closest matches.

In a RAG pipeline, the VectorDB is where all your document embeddings live. When a user asks something, their question becomes a vector and the VectorDB finds the most relevant document chunks.

Some options:
- **FAISS** — runs in memory, great for local projects
- **ChromaDB** — easy to set up, good for beginners
- **Pinecone** — cloud-hosted, scales well
- **Qdrant** — fast, open source

For a student project, FAISS or ChromaDB are the easiest to get started with.

In [ ]:
import numpy as np

# A tiny VectorDB written from scratch to show how it works
class VectorDB:
    def __init__(self):
        self.vecs = []
        self.docs = []

    def add(self, vec, doc):
        self.vecs.append(np.array(vec, dtype=float))
        self.docs.append(doc)

    def search(self, query, top=2):
        qv     = np.array(query, dtype=float)
        sims   = [
            np.dot(qv, v) / (np.linalg.norm(qv) * np.linalg.norm(v))
            for v in self.vecs
        ]
        ranked = sorted(zip(sims, self.docs), reverse=True)
        return ranked[:top]

db = VectorDB()
db.add([0.9, 0.1, 0.2], "Hostel location: Raiwind Rd, Lahore")
db.add([0.1, 0.9, 0.1], "Monthly rent: Rs. 12,000")
db.add([0.2, 0.1, 0.9], "Check-in at 12 PM, advance required")
db.add([0.8, 0.2, 0.1], "24-hour security and CCTV")

results = db.search([0.85, 0.15, 0.15], top=2)
print("Closest matches:")
for sim, doc in results:
    print(f"  [{sim:.3f}] {doc}")

## 7. Generative AI (GenAI)

Generative AI is AI that creates new content. That's the whole idea. Text, images, audio, video, code — if the model is producing something new rather than just classifying what already exists, it's generative.

The older kind of AI was mostly about prediction and classification: is this email spam or not? What number is in this image? Generative AI does something different — it makes things.

Examples by type:
- **Text** → ChatGPT, Claude, Gemini
- **Images** → DALL-E, Stable Diffusion, Midjourney
- **Code** → GitHub Copilot
- **Audio** → ElevenLabs, MusicLM
- **Video** → Sora, Runway

Under the hood, different architectures power different modalities. LLMs handle text. GANs and Diffusion models handle images. The term GenAI is just the umbrella over all of them.

In [ ]:
# Text generation — the most common form of GenAI in student projects
# pip install transformers torch

from transformers import pipeline

gen  = pipeline("text-generation", model="gpt2", max_new_tokens=60)
seed = "The future of AI in Pakistan is"
out  = gen(seed, do_sample=True, temperature=0.9)[0]["generated_text"]

print(out)

## 8. GANs — Generative Adversarial Networks

GANs were introduced in 2014 by Ian Goodfellow. The core idea is clever: train two networks against each other.

- **Generator** — takes random noise and tries to produce fake data that looks real
- **Discriminator** — looks at real and fake data and tries to tell them apart

They train at the same time. The Generator gets better at fooling the Discriminator. The Discriminator gets better at catching fakes. Eventually the Generator produces output good enough that the Discriminator can't reliably tell the difference.

```
Noise → [Generator] → Fake Image
                           ↓
Real Image   →   [Discriminator] → Real or Fake?
                           ↓
         Feedback goes back to both networks
```

GANs were the main tool for image generation for years. These days, Diffusion Models (like Stable Diffusion) tend to produce better results, but GANs are still important to understand. StyleGAN can generate human faces that look completely real — and the people in them do not exist.

In [ ]:
# Simple 1D GAN demo — Generator tries to match the real data distribution
import numpy as np

np.random.seed(0)

def real_data(n):
    # Real data is centered around 5
    return np.random.normal(loc=5.0, scale=0.5, size=n)

class Generator:
    def __init__(self):
        self.center = 0.0  # starts far from real data

    def generate(self, n):
        return np.random.normal(loc=self.center, scale=0.5, size=n)

    def update(self, real_mean):
        self.center += 0.5 * (real_mean - self.center)

class Discriminator:
    def score(self, samples, real_mean):
        # Returns what fraction it thinks looks real
        return np.mean(np.abs(samples - real_mean) < 1.0)

G = Generator()
D = Discriminator()
r = real_data(1000)
rm = r.mean()

print(f"Real data centered at: {rm:.2f}\n")
print(f"{'Epoch':<8} {'Generator mean':<18} {'Fool rate':<12}")
print("-" * 38)

for ep in range(1, 9):
    fake   = G.generate(500)
    fooled = D.score(fake, rm)
    print(f"{ep:<8} {fake.mean():<18.3f} {fooled*100:.1f}%")
    G.update(rm)

print("\nGenerator eventually matches the real distribution.")

---
## Summary

| Concept | What it actually is |
|---|---|
| **LangChain** | Framework that wires LLMs to tools, memory, and data |
| **LLM** | Model trained on massive text, predicts the next token |
| **RAG** | Adds a search step so the LLM can answer from your documents |
| **FAISS** | Fast vector search library — the search engine inside RAG |
| **Vector** | A list of numbers representing the meaning of text or data |
| **VectorDB** | Stores vectors and finds the closest ones to a query |
| **GenAI** | AI that creates new content (text, images, audio, code) |
| **GAN** | Two networks compete — one fakes, one judges — until fakes look real |

How they connect:
```
GenAI
 ├── Text:  LLM → RAG → VectorDB (FAISS) → all wired by LangChain
 └── Image: GANs or Diffusion Models

Vectors are the common thread — everything gets turned into numbers first.
```
